In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

ollama_api_key = os.getenv("OLLAMA_API_KEY")

print("OLLAMA API KEY FOUND:", bool(ollama_api_key))

OLLAMA API KEY FOUND: True


In [2]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests


**Tool create**

In [3]:
# creating tool

In [4]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/f130b4f6b3b6f86098d30b76/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [37]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [5]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1789948801,
 'time_last_update_utc': 'Mon, 21 Sep 2026 00:00:01 +0000',
 'time_next_update_unix': 1790035201,
 'time_next_update_utc': 'Tue, 22 Sep 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.0508}

In [6]:
convert.invoke({'base_currency_value':10, 'conversion_rate': 96.0508})

960.5079999999999

In [7]:
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv()

ollama_api_key = os.getenv("OLLAMA_API_KEY")

llm = ChatOllama(
    model="gpt-oss:120b",
    base_url="https://ollama.com",
    temperature=0.2,
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {ollama_api_key}"
        }
    }
)

In [8]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [9]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [10]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [11]:
ai_message = llm_with_tools.invoke(messages)

In [12]:
messages.append(ai_message)

In [13]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'bac50bad-d602-4d7e-b5b5-4055a7600e63',
  'type': 'tool_call'}]

In [14]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [54]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-09-21T11:10:42.88637509Z', 'done': True, 'done_reason': 'stop', 'total_duration': 985893386, 'load_duration': None, 'prompt_eval_count': 199, 'prompt_eval_duration': None, 'eval_count': 175, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--01a0c3a9-3931-75a3-b290-e49495327c66-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': '6a078a2f-d7ce-4ef8-adf8-ba60e6a07ca1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 199, 'output_tokens': 175, 'total_tokens': 374}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/doc

In [16]:
# Send the complete conversation back to the LLM
# Now the LLM can see the ToolMessage and generate the final answer
final_response = llm_with_tools.invoke(messages)

# Print the final answer
print(final_response.content)

The current conversion factor from Indian Rupees (INR) to U.S. Dollars (USD) is:

**1 INR = 0.01041 USD**

Using this rate:

\[
10 \text{ INR} \times 0.01041 \frac{\text{USD}}{\text{INR}} = 0.1041 \text{ USD}
\]

**So, 10 INR ≈ 0.104 USD (about 10.4 cents).**
